In [11]:
import pandas as pd
import numpy as np

df= pd.read_csv("../data/cleaned/merged_all.csv", parse_dates=["StartTimestamp"])

### Target, Features and processess flagging (0 or 1):

In [13]:
Target="OOS_Flag"
ids=["Batch_ID"]
meta_deta=["Product_ID", "Line_ID", "Operator", "StartTimestamp"]

specs_dict={
        'Mixing_Speed_RPM':(120,180),
        'Granulation_Temp_C':(55,70), 
        'Granulation_Moisture_%':(2.0,4.0),
        'Coating_Spray_Rate':(5,10), 
        'Inlet_Temp_C':(45,60)
       }
# process parameter flagging

for param, (lo, hi) in specs_dict.items():
    df[f"{param}_near_low"]= (df[param] <= lo+0.1*(hi-lo)).astype(int)
    df[f"{param}_near_high"]= (df[param] >= hi-0.1*(hi-lo)).astype(int)

# process step flagging

df["granulation_risk_flag"]=((df["Granulation_Temp_C"] > 66) | (df["Granulation_Moisture_%"] > 3.3)).astype(int)
df["coating_risk_flag"]=((df["Coating_Spray_Rate"] > 9) | (df["Inlet_Temp_C"] < 48)).astype(int)
df["mixing_risk_flag"]=((df["Mixing_Speed_RPM"] < 130)).astype(int)

### Rolling Standard Deviations (Stability/Variability check per line)

In [14]:
df = df.sort_values("StartTimestamp").set_index("StartTimestamp")

for param in ["Mixing_Speed_RPM","Granulation_Temp_C","Granulation_Moisture_%","Coating_Spray_Rate"]:
    df[f"{param}_roll_std"] = (
        df.groupby("Line_ID")[param]
          .rolling(window=10, min_periods=5).std()
          .reset_index(level=0, drop=True)
    )

df.bfill(inplace=True)

### Product ID Dummies

In [15]:
df = pd.get_dummies(df, columns=["Product_ID"], drop_first=True)
df.head(20)

,Batch_ID,Line_ID,Operator,Blender_load_%,Coating_Spray_Rate,Coating_Time_Min,Granulation_Moisture_%,Granulation_Temp_C,Granulation_Time_Min,Inlet_Temp_C,...,Inlet_Temp_C_near_high,granulation_risk_flag,coating_risk_flag,mixing_risk_flag,Mixing_Speed_RPM_roll_std,Granulation_Temp_C_roll_std,Granulation_Moisture_%_roll_std,Coating_Spray_Rate_roll_std,Product_ID_GastroEase_CR,Product_ID_NeuroCalm_IR
StartTimestamp,,,,,,,,,,,,,,,,,,,,,
2023-01-01 06:00:00,BATCH_0001,Line_2,Operator_A,64.1951,7.7535,32.0761,2.4827,62.5640,20.5537,59.3939,...,1,0,0,1,24.920830,4.267318,0.781768,1.133414,False,False
2023-01-01 16:00:00,BATCH_0002,Line_2,Operator_B,70.0006,7.6885,32.9209,2.8915,63.7867,18.9745,56.7596,...,0,0,0,1,24.920830,4.267318,0.781768,1.133414,False,False
2023-01-02 00:00:00,BATCH_0003,Line_2,Operator_B,56.6179,10.4212,38.8662,3.8597,68.1738,22.3567,58.4942,...,0,1,1,0,24.920830,4.267318,0.781768,1.133414,False,False
2023-01-02 10:00:00,BATCH_0004,Line_2,Operator_D,60.5145,8.0295,31.1342,2.7301,57.3226,24.3476,53.6096,...,0,0,0,0,24.920830,4.267318,0.781768,1.133414,False,True
2023-01-02 20:00:00,BATCH_0005,Line_1,Operator_A,69.7215,8.4464,36.7114,2.6197,70.1598,14.1825,48.8420,...,0,1,0,0,24.920830,4.267318,0.781768,1.133414,False,True
2023-01-03 04:00:00,BATCH_0006,Line_2,Operator_D,74.3854,8.6017,30.3424,1.6902,58.9251,28.2223,51.1206,...,0,0,0,1,24.920830,4.267318,0.781768,1.133414,False,True
2023-01-03 12:00:00,BATCH_0007,Line_2,Operator_C,73.6119,6.6225,33.4261,2.8200,65.3614,16.0569,49.2400,...,0,0,0,0,24.793688,4.035109,0.700181,1.270631,False,False
2023-01-03 22:00:00,BATCH_0008,Line_2,Operator_C,70.0358,8.7691,35.3995,4.0790,81.8311,29.0039,51.6456,...,0,1,0,0,22.663493,8.118774,0.813941,1.180663,False,False
2023-01-04 07:00:00,BATCH_0009,Line_1,Operator_A,67.7749,8.5556,25.2483,3.4598,59.7165,24.4383,66.9509,...,1,1,0,0,21.364218,7.851296,0.786931,1.202725,False,False
